# Assignment 2

**This was supposed to be 2 assignments but is a big assignment. So go slow and learn what you are doing, have fun.**


In [26]:
!pip install langchain langchain-community langchain-text-splitters
!pip install sentence-transformers chromadb
!pip install langchain-groq pymupdf
!pip install python-dotenv

In [27]:
from dotenv import load_dotenv
import os

load_dotenv()

from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)

# Part 1: Data Ingestion

Data Ingestion is the first step in a RAG pipeline. It involves collecting, reading, and loading raw data from various sources (such as PDFs, HTML, or databases) into the system. Here, we read all PDF files in a given directory and parse their content into structured documents containing page content and metadata.


Here we are doing only for PDF but in the final project we will do for pdf, csv, xlsx, docx, txt.
If you want, you can practice extracting data from one more file format.

In [3]:
import os
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader

/tmp/ipykernel_16113/1982408242.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [5]:
def process_all_pdfs(pdf_directory):
    all_documents = []

    # Find all PDFs
    pdf_files = list(Path(pdf_directory).glob('**/*.pdf'))
    print(f"Found {len(pdf_files)} PDF file(s) in '{pdf_directory}'")

    for pdf_file in pdf_files:
        print(f"  Loading: {pdf_file.name}")

        docs = PyPDFLoader(str(pdf_file)).load()

        for doc in docs:
            doc.metadata['source_file'] = pdf_file.name
            doc.metadata['file_type'] = 'pdf'

        all_documents.extend(docs)

    print(f"Total pages loaded: {len(all_documents)}")
    return all_documents


os.makedirs('./data/pdfs', exist_ok=True)
all_pdf_documents = process_all_pdfs('./data/pdfs')
print(all_pdf_documents)

Found 1 PDF file(s) in './data/pdfs'
  Loading: Image_fundamentals.pdf
Total pages loaded: 12
[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-12-10T22:47:54+00:00', 'author': '', 'keywords': '', 'moddate': '2025-12-10T22:47:54+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data/pdfs/Image_fundamentals.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'Image_fundamentals.pdf', 'file_type': 'pdf'}, page_content='Image Fundamentals\nDecember 10, 2025\nContents\n1 Introduction 2\n2 Colour and Colour Spaces 2\n2.1 Colour . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 2\n2.2 Colour Spaces . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 2\n2.2.1 RGB Space . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 

# Part 2: Chunking

Chunking is the process of breaking down large documents into smaller, cohesive segments (chunks). Since LLMs have a limited context window and retrieve information more accurately from smaller context blocks, we split large documents. We use **RecursiveCharacterTextSplitter** with overlap to maintain context across chunk boundaries.


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [7]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,   # 20% overlap keeps boundary context
        length_function=len,
        separators=['\n\n', '\n', ' ', '']  # tries paragraph → line → word → char
    )

    chunks = splitter.split_documents(documents)
    print(f"{len(documents)} document page(s) → {len(chunks)} chunk(s)")
    return chunks


chunks = split_documents(all_pdf_documents)
chunks

12 document page(s) → 25 chunk(s)


[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-12-10T22:47:54+00:00', 'author': '', 'keywords': '', 'moddate': '2025-12-10T22:47:54+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data/pdfs/Image_fundamentals.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'Image_fundamentals.pdf', 'file_type': 'pdf'}, page_content='Image Fundamentals\nDecember 10, 2025\nContents\n1 Introduction 2\n2 Colour and Colour Spaces 2\n2.1 Colour . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 2\n2.2 Colour Spaces . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 2\n2.2.1 RGB Space . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 2\n2.2.2 HSI colour space . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 4\n3

# Part 3: Embedding

Embedding converts text chunks into numerical vectors that capture semantic meaning. Sentences that are semantically similar end up close together in vector space. We use the pretrained `all-MiniLM-L6-v2` model from sentence-transformers.


In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [9]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        print(f"Loading embedding model: {self.model_name} ...")
        self.model = SentenceTransformer(self.model_name)
        # embedding_dimension tells us the vector size (384 for MiniLM)
        dim = self.model.get_sentence_embedding_dimension()
        print(f"Model loaded. Embedding dimension: {dim}")

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.
        Returns numpy array of shape (len(texts), embedding_dim)
        """
        if self.model is None:
            raise RuntimeError("Model not loaded. Call _load_model() first.")
        return self.model.encode(texts, show_progress_bar=True)


embedding_manager = EmbeddingManager()

Loading embedding model: all-MiniLM-L6-v2 ...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded. Embedding dimension: 384


/tmp/ipykernel_16113/1387062205.py:14: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = self.model.get_sentence_embedding_dimension()


# Part 4: Vector DB

A Vector Database stores high-dimensional embeddings and lets us do fast nearest-neighbour search. We use ChromaDB, which runs locally and persists to disk automatically.


In [10]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents",
                 persist_directory: str = "./data/vector_store"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        os.makedirs(self.persist_directory, exist_ok=True)

        self.client = chromadb.PersistentClient(path=self.persist_directory)

        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "DocuMind PDF document store"}
        )
        print(f"ChromaDB collection '{self.collection_name}' ready. "
              f"Documents already stored: {self.collection.count()}")

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents and their embeddings to the vector store"""
        if len(documents) != len(embeddings):
            raise ValueError(f"Mismatch: {len(documents)} docs vs {len(embeddings)} embeddings")

        ids, emb_list, meta_list, content_list = [], [], [], []

        for i, (doc, emb) in enumerate(zip(documents, embeddings)):
            # Short unique id for each chunk
            uid = uuid.uuid4().hex[:8]

            # Merge original metadata with extra fields
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)

            ids.append(uid)
            emb_list.append(emb.tolist())   # ChromaDB needs plain Python list
            meta_list.append(metadata)
            content_list.append(doc.page_content)

        self.collection.add(
            ids=ids,
            embeddings=emb_list,
            metadatas=meta_list,
            documents=content_list
        )
        print(f"Added {len(documents)} chunks. Total in store: {self.collection.count()}")


vectorstore = VectorStore()

ChromaDB collection 'pdf_documents' ready. Documents already stored: 0


In [11]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2025-12-10T22:47:54+00:00', 'author': '', 'keywords': '', 'moddate': '2025-12-10T22:47:54+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'data/pdfs/Image_fundamentals.pdf', 'total_pages': 12, 'page': 0, 'page_label': '1', 'source_file': 'Image_fundamentals.pdf', 'file_type': 'pdf'}, page_content='Image Fundamentals\nDecember 10, 2025\nContents\n1 Introduction 2\n2 Colour and Colour Spaces 2\n2.1 Colour . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 2\n2.2 Colour Spaces . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 2\n2.2.1 RGB Space . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 2\n2.2.2 HSI colour space . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 4\n3

## Convert text to embeddings and store

In [12]:
texts = [doc.page_content for doc in chunks]
print(f"{len(texts)} text chunk(s) ready for embedding")
texts

25 text chunk(s) ready for embedding


['Image Fundamentals\nDecember 10, 2025\nContents\n1 Introduction 2\n2 Colour and Colour Spaces 2\n2.1 Colour . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 2\n2.2 Colour Spaces . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 2\n2.2.1 RGB Space . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 2\n2.2.2 HSI colour space . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 4\n3 Digital images 4\n3.1 Binary Image . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 4\n3.2 Grayscale Image . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 6\n3.3 RGB Image . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 6\n3.4 HSI Image . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 6\n4 Pixel Level Operations 8\n4.1 Greyscaling . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 8'

In [13]:
embeddings = embedding_manager.generate_embeddings(texts)
print(f"Embeddings shape: {embeddings.shape}")

# Store in ChromaDB
vectorstore.add_documents(chunks, embeddings)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embeddings shape: (25, 384)
Added 25 chunks. Total in store: 25


# Part 5: Query Retrieval
# Part 6: Similarity Search

The user query is embedded into the same vector space. ChromaDB finds the nearest neighbours by L2 distance. We convert distance → similarity score (1 - distance) and filter by a threshold.


In [14]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5,
                 score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query.
        Returns list of dicts with keys:
          id, content, metadata, similarity_score, distance, rank
        """
        # 1. Embed the query (returns shape (1, dim) — we need the inner list)
        query_embedding = self.embedding_manager.generate_embeddings([query])

        # 2. Query ChromaDB — returns distances (L2), not similarities
        results = self.vector_store.collection.query(
            query_embeddings=query_embedding.tolist(),
            n_results=top_k
        )

        # ChromaDB wraps results in an extra list (batch dimension)
        ids       = results['ids'][0]
        documents = results['documents'][0]
        metadatas = results['metadatas'][0]
        distances = results['distances'][0]

        # 3 & 4. Convert distance → similarity and filter
        retrieved = []
        for rank, (rid, content, meta, dist) in enumerate(
                zip(ids, documents, metadatas, distances), start=1):
            similarity_score = 1 - dist   # ChromaDB uses L2; lower dist = more similar
            if similarity_score >= score_threshold:
                retrieved.append({
                    'id': rid,
                    'content': content,
                    'metadata': meta,
                    'similarity_score': round(similarity_score, 4),
                    'distance': round(dist, 4),
                    'rank': rank
                })

        return retrieved


rag_retriever = RAGRetriever(vectorstore, embedding_manager)
rag_retriever

In [16]:
# Test retrieval — change the query to something relevant to your PDF
results = rag_retriever.retrieve("how does grayscaling work", top_k=3)
for r in results:
    print(f"Rank {r['rank']} | Score: {r['similarity_score']} | {r['content'][:120]}...")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Rank 1 | Score: 0.2153 | 4 Pixel Level Operations
Pixel-level operations are image processing techniques that work by directly modifying the
nume...
Rank 2 | Score: 0.1409 | Using built in cv2 function
•Using the cv2.cvtColor() function
•Using the cv2.imread() function with flag=zero-In this m...
Rank 3 | Score: 0.0807 | to grayscale level
Together, hue + saturation form the chromatic (color-carrying) information, while intensity
carries t...


# Part 7: Prompting and Calling LLM

We take the retrieved contexts, format them into a structured prompt alongside the user's query, and pass to the LLM to generate a grounded response.


In [20]:
def rag_simple(query, retriever, llm, top_k=3):
    """Basic RAG: retrieve → format prompt → call LLM → return answer string"""
    # 1. Retrieve relevant chunks
    docs = retriever.retrieve(query, top_k=top_k)

    # 2. Join chunk contents into one context block
    context = "\n\n".join([d['content'] for d in docs])

    # 3. Format the prompt — instruct the LLM to stay grounded in context
    prompt = f"""You are a helpful assistant. Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't have enough information to answer this."

Context:
{context}

Question: {query}

Answer:"""

    # 4. Call the LLM and return the string content
    response = llm.invoke(prompt)
    return response.content

#answer = rag_simple("three reasons for forgetting", rag_retriever, llm)
answer = rag_simple("what is adaptive thresholding", rag_retriever, llm)
print(answer)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Adaptive thresholding addresses one of the main limitations of simple (global) thresholding its inability to handle images with varying lighting conditions. Instead of using a single global threshold value for the whole image, adaptive thresholding calculates the threshold for small regions around each pixel.


# Part 8: Advanced RAG

Advanced RAG adds citations, confidence scoring, streaming simulation, summarization, and conversation history.


In [23]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    docs = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)

    if not docs:
        return {
            'answer': "No relevant documents found above the score threshold.",
            'sources': [],
            'confidence': 0.0
        }

    sources = []
    for d in docs:
        sources.append({
            'source_file': d['metadata'].get('source_file', 'unknown'),
            'page': d['metadata'].get('page', 'N/A'),
            'similarity_score': d['similarity_score'],
            'content_preview': d['content'][:100] + '...'
        })

    confidence = max(d['similarity_score'] for d in docs)
    context = "\n\n".join([d['content'] for d in docs])
    prompt = f"""You are a helpful assistant. Answer using ONLY the context below.

Context:
{context}

Question: {query}

Answer:"""
    answer = llm.invoke(prompt).content

    result = {
        'answer': answer,
        'sources': sources,
        'confidence': round(confidence, 4)
    }
    if return_context:
        result['context'] = context
    return result

In [25]:
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2,
              stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # 1. Retrieve
        docs = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)

        # guard — prevent hallucination on empty retrieval
        if not docs:
            return {
                'question': question,
                'answer': "No relevant documents found above the score threshold.",
                'sources': [],
                'summary': None,
                'history': self.history
            }

        # 2. Build context and sources
        context_parts = []
        sources = []
        for i, d in enumerate(docs, start=1):
            context_parts.append(f"[{i}] {d['content']}")
            sources.append({
                'rank': i,
                'source_file': d['metadata'].get('source_file', 'unknown'),
                'page': d['metadata'].get('page', 'N/A'),
                'score': d['similarity_score']
            })
        context = "\n\n".join(context_parts)

        prompt = f"""You are a helpful assistant. Answer using ONLY the provided context.
Reference sources by their number [1], [2], etc.

Context:
{context}

Question: {question}

Answer:"""

        # 3. Simulate streaming if requested
        if stream:
            print("[Streaming response...]")
            for word in prompt.split():
                print(word, end=' ', flush=True)
                time.sleep(0.01)
            print()

        # 4. Generate answer
        answer = self.llm.invoke(prompt).content

        # 5. Append citations
        citations = "\n\nSources:"
        for s in sources:
            citations += f"\n  [{s['rank']}] {s['source_file']} (page {s['page']}, score {s['score']})"
        answer_with_citations = answer + citations

        # 6. Optional summarization
        summary = None
        if summarize:
            summary_prompt = f"Summarize this answer in 2 sentences:\n\n{answer}"
            summary = self.llm.invoke(summary_prompt).content

        # 7. Store in history
        self.history.append({
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary
        })

        # 8. Return
        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }


# --- test ---
pipeline = AdvancedRAGPipeline(rag_retriever, llm)
result = pipeline.query("3 reasons of forgetting", summarize=True)
print(result['answer'])
print("\nSummary:", result['summary'])

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

No relevant documents found above the score threshold.

Summary: None
